# Proyecto 12 – Experimentos para mejorar el modelo final
Objetivo: mejorar PR-AUC y precision@K del HistGradientBoosting sin aumentar los falsos positivos.

**Reglas del experimento**
1. Todo se compara y se elige en **valid**. Test se evalúa **una sola vez**, al final (sección 5).
2. La API no cambia: las entradas siguen siendo `V.COLUMNAS_API`.
3. Género, edad y ocupación no entran al modelo.

Cada configuración se entrena con 3 semillas (42, 7, 123) y se reporta el promedio: solo la semilla mueve el PR-AUC en ±0,007, así que una diferencia menor que eso no se considera mejora. En el bloque B un cambio solo reemplaza al vigente si mejora el PR-AUC en más de 0,002.

`variables.py` ya incluye las variables nuevas (v1.1.0); los scripts usan `construir_variables_v1` y `NUM_V1` (en `exp_comun.py`) para seguir comparando contra el modelo 1.0.0.

Los scripts que generaron los resultados están en `experimentos/` (`exp_a.py` … `exp_final.py`). Este notebook lee sus resultados y repite la evaluación final en test.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
BASE = Path("C:/Users/isalo/Desktop/universidad/9.Semestre_9/Big data/Proyecto Final")
EXP = BASE / "experimentos"
sys.path.insert(0, str(EXP))
from exp_comun import *   # variables_v2, pipeline, métricas, umbral
cols = ["exp", "pr_auc", "pr_auc_std", "p@100", "p@500", "p@1000", "fp@rec90", "fp@rec95"]

## 1. Bloque A – Variables derivadas (sin cambiar la API)
Todas se calculan dentro del pipeline a partir de los campos que ya envía el backend.

| Variable | Qué mide |
|---|---|
| `log_monto_promedio_30d` | Nivel de gasto habitual del cliente |
| `log_monto_promedio_categoria` | Nivel de gasto habitual del cliente en esa categoría |
| `diferencia_monto_30d` | monto − promedio 30 días (en pesos) |
| `ratio_categoria_vs_30d` | Qué tan cara es esa categoría para ese cliente |
| `sin_historial` | 1 si `segundos_desde_ultima` es el valor por defecto |
| `hora_decimal` | Hora con minutos |

In [2]:
pd.read_csv(EXP / "res_a.csv")[cols].round(4)

,exp,pr_auc,pr_auc_std,p@100,p@500,p@1000,fp@rec90,fp@rec95
0,actual,0.9195,0.0068,1.0,0.9973,0.9647,453.3333,1163.3333
1,A: todas,0.9271,0.0026,1.0,1.0000,0.9820,406.0000,1069.0000
2,A: todas menos log_monto_promedio_30d,0.9246,0.0071,1.0,0.9993,0.9740,413.6667,1076.3333
3,A: todas menos log_monto_promedio_categoria,0.9242,0.0019,1.0,1.0000,0.9760,433.6667,1147.6667
4,A: todas menos diferencia_monto_30d,0.9272,0.0031,1.0,1.0000,0.9770,390.0000,1124.0000
5,A: todas menos ratio_categoria_vs_30d,0.9252,0.0015,1.0,0.9987,0.9750,409.3333,1066.3333
6,A: todas menos sin_historial,0.9281,0.0029,1.0,1.0000,0.9817,394.6667,1050.0000
7,A: todas menos hora_decimal,0.9253,0.0049,1.0,0.9993,0.9780,423.0000,1064.0000
8,A: actual + log_monto_promedio_30d,0.9205,0.0034,1.0,0.9967,0.9693,426.6667,1230.0000
9,A: actual + log_monto_promedio_categoria,0.9226,0.0035,1.0,0.9967,0.9667,418.0000,1117.3333


**Resultado:** se quedan 4 variables: `log_monto_promedio_30d`, `log_monto_promedio_categoria`, `ratio_categoria_vs_30d` y `hora_decimal` (quitar cualquiera baja el PR-AUC). `sin_historial` y `diferencia_monto_30d` se descartan: no ayudan o empeoran un poco.

## 2. Bloque B – Hiperparámetros (con las 4 variables nuevas)
Búsqueda por etapas: peso de clases → tasa de aprendizaje / hojas / iteraciones → regularización → categorías nativas.

In [3]:
pd.read_csv(EXP / "res_b.csv")[cols].round(4)

,exp,pr_auc,pr_auc_std,p@100,p@500,p@1000,fp@rec90,fp@rec95
0,B1 peso=balanced,0.9274,0.0031,1.0000,0.9993,0.9777,388.3333,1098.6667
1,B1 peso=None,0.8033,0.0128,0.9367,0.9367,0.9470,3498.6667,55695.0000
2,B1 peso=1:10,0.9069,0.0194,0.9300,0.9820,0.9797,441.6667,1174.6667
3,B1 peso=1:30,0.9281,0.0103,0.9833,0.9967,0.9860,376.0000,1075.3333
4,B2 lr=0.05 iter=800,0.9231,0.0029,1.0000,1.0000,0.9740,411.0000,1111.3333
5,B2 lr=0.1 iter=800,0.9274,0.0031,1.0000,0.9993,0.9777,388.3333,1098.6667
6,B2 hojas=15 iter=800,0.9265,0.0023,1.0000,1.0000,0.9790,403.6667,1103.6667
7,B2 hojas=63 iter=800,0.9251,0.0002,1.0000,0.9980,0.9757,405.6667,1026.0000
8,B2 hojas=127 lr=0.05 iter=800,0.9167,0.0039,0.9900,0.9913,0.9627,427.3333,1238.3333
9,B3 min_leaf=100,0.9257,0.0025,1.0000,0.9987,0.9750,405.3333,1094.3333


**Resultado:** la configuración original (`learning_rate=0.1`, 31 hojas, `class_weight="balanced"`) sigue siendo la mejor; ninguna alternativa la supera por encima del ruido. Sin peso de clases el PR-AUC cae a 0,80 (el early stopping se detiene a las 12 iteraciones). Toda la mejora viene de las variables del bloque A.

## 3. Bloque C – Criterio para elegir el umbral
El umbral se elige en la **primera mitad** (temporal) de valid y se mide en la **segunda mitad**, para ver qué criterio se sostiene cuando cambia la tasa de fraude (entre valid y test baja de 0,59 % a 0,39 %).

In [4]:
pd.set_option("display.width", 250)
c = pd.read_csv(EXP / "res_c.csv")
c[["modelo", "criterio", "umbral", "precision_v1", "precision_v2", "recall_v1", "recall_v2",
   "falsos_positivos_v1", "falsos_positivos_v2", "tasa_fp_v1", "tasa_fp_v2"]].round(4)

,modelo,criterio,umbral,precision_v1,precision_v2,recall_v1,recall_v2,falsos_positivos_v1,falsos_positivos_v2,tasa_fp_v1,tasa_fp_v2
0,actual,precision>=0.50,0.7846,0.5004,0.5616,0.9547,0.9531,673,619,0.0052,0.0048
1,actual,recall>=0.95,0.8373,0.5634,0.6132,0.9504,0.9471,520,497,0.0040,0.0039
2,actual,tasa_fp<=0.004,0.8394,0.5649,0.6136,0.9490,0.9447,516,495,0.0040,0.0038
3,actual,tasa_fp<=0.002,0.9296,0.7143,0.7388,0.9136,0.9147,258,269,0.0020,0.0021
4,actual,tasa_alertas=0.009,0.8442,0.5724,0.6196,0.9462,0.9435,499,482,0.0039,0.0037
5,mejorado,precision>=0.50,0.7310,0.5000,0.5463,0.9575,0.9579,676,662,0.0052,0.0051
6,mejorado,recall>=0.95,0.8287,0.5907,0.6332,0.9504,0.9483,465,457,0.0036,0.0035
7,mejorado,tasa_fp<=0.004,0.8070,0.5657,0.6126,0.9518,0.9483,516,499,0.0040,0.0039
8,mejorado,tasa_fp<=0.002,0.9262,0.7171,0.7539,0.9263,0.9207,258,250,0.0020,0.0019
9,mejorado,tasa_alertas=0.009,0.8164,0.5758,0.6222,0.9518,0.9483,495,479,0.0038,0.0037


**Resultado:** el criterio actual (precisión ≥ 0,50) genera más falsos positivos (tasa ≈ 0,52 %) y depende de la tasa de fraude, que cambia en el tiempo. Fijar la **tasa de falsos positivos** (`tasa_fp ≤ 0,4 %`) solo depende de las transacciones normales, por eso se sostiene igual en la segunda mitad (0,40 % → 0,39 %), con recall ≈ 95 %. Se elige `tasa_fp ≤ 0,004`. `tasa_fp ≤ 0,002` se reporta como punto de operación alternativo (la mitad de alertas, recall ≈ 92 %).

## 4. Bloque D – Variables que el backend tendría que calcular (SOLO medición)
Estas variables **no se agregan al modelo**: solo se mide cuánto aportarían. Se calculan con transacciones anteriores del mismo cliente (ventanas `closed="left"`).

| Variable | Definición |
|---|---|
| `tx_1h`, `tx_24h` | Nº de transacciones del cliente en la última hora / 24 h |
| `monto_24h` | Suma de montos del cliente en las últimas 24 h |
| `tx_previas` | Nº total de transacciones previas del cliente |
| `zscore_30d` | (monto − promedio 30 d) / desviación estándar 30 d |

In [5]:
pd.read_csv(EXP / "res_d.csv")[cols].round(4)

,exp,pr_auc,pr_auc_std,p@100,p@500,p@1000,fp@rec90,fp@rec95
0,D: actual + todas D,0.9726,0.0007,1.0,0.9987,0.9980,76.0000,228.6667
1,D: A elegidas + todas D,0.9697,0.0055,1.0,0.9993,0.9963,90.0000,286.0000
2,D: A elegidas + tx_1h,0.9273,0.0019,1.0,1.0000,0.9797,386.3333,1074.6667
3,D: A elegidas + tx_24h,0.9293,0.0013,1.0,0.9987,0.9823,363.6667,1203.0000
4,D: A elegidas + monto_24h,0.9633,0.0017,1.0,0.9960,0.9947,133.0000,423.0000
5,D: A elegidas + tx_previas,0.9304,0.0017,1.0,1.0000,0.9843,407.0000,1052.0000
6,D: A elegidas + zscore_30d,0.9300,0.0036,1.0,0.9980,0.9750,361.3333,939.3333


**Resultado:** `monto_24h` es por mucho la más valiosa: sola sube el PR-AUC de 0,927 a 0,963 y reduce los falsos positivos con recall 90 % de ~390 a ~133. Con las 5 variables D se llega a 0,973. Las demás aportan poco por separado. Agregarla exige que el backend calcule la suma de montos del cliente en las últimas 24 h (ampliaría `COLUMNAS_API`), por eso queda como propuesta.

> **Decisión (v1.1.0): `monto_24h` se descarta para esta versión.** Usarla requiere cambiar el contrato con el backend (agregar un campo a `COLUMNAS_API` y que NestJS calcule la suma de montos de las últimas 24 h). La API se mantiene igual; queda como candidata para una versión futura.

## 5. Evaluación final en test (única)
Modelos entrenados con train (semilla 42), umbral elegido solo con valid. Las filas 1 y 2 son el mismo modelo actual con dos criterios de umbral, para separar el efecto del umbral del efecto de las variables.

In [6]:
%run -i "C:/Users/isalo/Desktop/universidad/9.Semestre_9/Big data/Proyecto Final/experimentos/exp_final.py" "tasa_fp<=0.004"
res.round(4).T

modelo            Actual (umbral precisión≥0,50)  Actual (umbral tasa_fp<=0.004)  Mejorado (umbral tasa_fp<=0.004)  Mejorado (umbral tasa_fp<=0.002, punto alternativo)
umbral                                    0.7452                          0.8348                            0.8021                                               0.9241
pr_auc                                    0.8825                          0.8825                            0.8920                                               0.8920
p@100                                     1.0000                          1.0000                            1.0000                                               1.0000
p@500                                     1.0000                          1.0000                            1.0000                                               1.0000
p@1000                                    0.9870                          0.9870                            0.9910                                              

modelo,"Actual (umbral precisión≥0,50)",Actual (umbral tasa_fp<=0.004),Mejorado (umbral tasa_fp<=0.004),"Mejorado (umbral tasa_fp<=0.002, punto alternativo)"
umbral,0.7452,0.8348,0.8021,0.9241
pr_auc,0.8825,0.8825,0.8920,0.8920
p@100,1.0000,1.0000,1.0000,1.0000
p@500,1.0000,1.0000,1.0000,1.0000
p@1000,0.9870,0.9870,0.9910,0.9910
fp@rec90,1404.0000,1404.0000,1160.0000,1160.0000
fp@rec95,3877.0000,3877.0000,3447.0000,3447.0000
precision,0.4033,0.4867,0.4850,0.6456
recall,0.9413,0.9277,0.9319,0.8918
falsos_positivos,2987.0000,2099.0000,2123.0000,1050.0000


## 6. Conclusiones
- **Variables nuevas (sin cambiar la API):** PR-AUC en test 0,8825 → 0,8920; falsos positivos con recall 90 % 1.404 → 1.160 (−17 %); precision@1000 0,987 → 0,991. precision@100 y @500 ya estaban en 1,00 y siguen igual.
- **Umbral por tasa de falsos positivos** (`tasa_fp ≤ 0,4 %` en valid): falsos positivos en test 2.987 → 2.123 (−29 %) y precisión 0,40 → 0,49, con recall 0,941 → 0,932.
- **Hiperparámetros:** la configuración original se mantiene.
- **Con apoyo del backend:** `monto_24h` llevaría el PR-AUC en valid a ≈ 0,96–0,97, pero **se descartó para la v1.1.0** porque requiere cambiar el contrato con el backend.

**Aplicado en la versión 1.1.0** (`variables.py` y `02_modelo.ipynb`): las 4 variables del bloque A, la configuración original de hiperparámetros y el umbral `tasa_fp ≤ 0,002` elegido en valid (umbral 0,9241; en test: precisión 0,646, recall 0,892 y 1.050 falsos positivos). Es el punto que cumple precisión ≥ 0,50 y recall ≥ 0,80 con la mitad de alertas.